# GoPay Google Play Review - Sentiment Analysis

**Author:** Muhammad Razan Parisya Putra  
**Notebook:** `04 - Sentiment Analysis`

This notebook continues from the preprocessing covered in [3-Gopay-Review-Preprocessing.ipynb](https://colab.research.google.com/drive/1Iwg8LQ69nvhEtv6wE4Ou7uM2hySBrhKd?usp=sharing)

---

## Objective

Train and evaluate multiple machine learning classifiers to predict sentiment (positive / neutral / negative) from preprocessed GoPay review text.

## Pipeline

| Step | Task |
|------|------|
| 1 | Setup & Load Data |
| 2 | Exploratory Check |
| 3 | Feature Extraction (TF-IDF) |
| 4 | Train-Test Split |
| 5 | Model Training (5 classifiers) |
| 6 | Model Evaluation & Comparison |
| 7 | Best Model — Detailed Analysis |
| 8 | Save Results |

## 1. Setup & Installation

In [ ]:
!pip install xgboost -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import ast
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
print('All libraries loaded successfully.')

## 2. Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Tugas 1/Dataset/gopay_reviews_clean.csv')

# tokens_stemmed was saved as string — convert back to list
df['tokens_stemmed'] = df['tokens_stemmed'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

## 3. Exploratory Check

In [ ]:
# Sentiment distribution
sent_counts = df['sentiment'].value_counts()
print('Sentiment distribution:')
print(sent_counts)
print()
print('Class proportions (%)')
print((sent_counts / len(df) * 100).round(2))

In [ ]:
# Visualize sentiment distribution
colors_sent = {'positive': '#388e3c', 'neutral': '#ffb74d', 'negative': '#d32f2f'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax = axes[0]
bars = ax.bar(sent_counts.index, sent_counts.values,
              color=[colors_sent[s] for s in sent_counts.index], edgecolor='white')
ax.set_ylabel('Number of Reviews')
ax.set_title('Sentiment Label Distribution')
for bar, val in zip(bars, sent_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + len(df)*0.003,
            f'{val:,}', ha='center', fontweight='bold')

# Pie chart
ax2 = axes[1]
ax2.pie(
    sent_counts.values,
    labels=sent_counts.index,
    colors=[colors_sent[s] for s in sent_counts.index],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
ax2.set_title('Sentiment Proportion')

plt.suptitle('Sentiment Distribution — GoPay Reviews', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Missing check
print('Missing values in key columns:')
print(df[['final_text', 'sentiment']].isnull().sum())

# Drop any remaining NaN in final_text
df = df.dropna(subset=['final_text'])
df = df[df['final_text'].str.strip() != '']
print(f'\nClean dataset size: {len(df):,} rows')

## 4. Feature Extraction — TF-IDF

We convert the `final_text` column (preprocessed, stemmed Indonesian text) into a numeric matrix using **TF-IDF** (Term Frequency–Inverse Document Frequency).  
TF-IDF assigns higher weight to words that are frequent in a document but rare across the corpus — ideal for distinguishing sentiment-carrying words.

| Parameter | Value | Reason |
|-----------|-------|---------|
| `max_features` | 10,000 | Limit vocabulary to most informative words |
| `ngram_range` | (1,2) | Include bigrams to capture phrases like "tidak bagus" |
| `min_df` | 3 | Ignore very rare words (noise) |
| `sublinear_tf` | True | Apply log normalization to term frequency |

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
    sublinear_tf=True
)

X = tfidf.fit_transform(df['final_text'])
y = df['sentiment']

print(f'TF-IDF matrix shape : {X.shape}')
print(f'Vocabulary size     : {len(tfidf.vocabulary_):,}')
print(f'Target classes      : {sorted(y.unique())}')

## 5. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y          # preserve class proportions
)

print(f'Training set   : {X_train.shape[0]:,} samples ({X_train.shape[0]/len(df)*100:.0f}%)')
print(f'Test set       : {X_test.shape[0]:,} samples ({X_test.shape[0]/len(df)*100:.0f}%)')
print()
print('Class distribution in training set:')
print(pd.Series(y_train).value_counts())
print()
print('Class distribution in test set:')
print(pd.Series(y_test).value_counts())

## 6. Label Encoding (for XGBoost)

XGBoost requires integer labels. We encode: `negative=0`, `neutral=1`, `positive=2`.

In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

print('Label encoding mapping:')
for cls, idx in zip(le.classes_, le.transform(le.classes_)):
    print(f'  {cls} → {idx}')

## 7. Model Training

We train **5 classifiers** commonly used for text classification:

| Model | Key Strength |
|-------|--------------|
| Linear SVM | Best for high-dimensional sparse text |
| Logistic Regression | Fast, interpretable baseline |
| Naive Bayes (Multinomial) | Efficient for TF-IDF features |
| XGBoost | Gradient boosting, handles imbalance |
| Random Forest | Ensemble, robust to noise |

In [ ]:
from sklearn.pipeline import Pipeline
import time

# Define models
models = {
    'Linear SVM': LinearSVC(C=1.0, max_iter=2000, random_state=RANDOM_STATE),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs',
                                               multi_class='multinomial', random_state=RANDOM_STATE),
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                             use_label_encoder=False, eval_metric='mlogloss',
                             random_state=RANDOM_STATE, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=20,
                                            random_state=RANDOM_STATE, n_jobs=-1)
}

results = {}

for name, model in models.items():
    print(f'Training {name}...', end=' ')
    start = time.time()

    # XGBoost needs encoded labels
    if name == 'XGBoost':
        model.fit(X_train, y_train_enc)
        y_pred_enc = model.predict(X_test)
        y_pred = le.inverse_transform(y_pred_enc)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    elapsed = time.time() - start
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)

    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'accuracy': acc,
        'report': report,
        'time': elapsed
    }

    print(f'Accuracy: {acc:.4f} | Time: {elapsed:.1f}s')

print('\nAll models trained!')

## 8. Model Comparison

In [ ]:
# Build summary DataFrame
summary_rows = []
for name, res in results.items():
    report = res['report']
    summary_rows.append({
        'Model': name,
        'Accuracy': round(res['accuracy'], 4),
        'Precision (macro)': round(report['macro avg']['precision'], 4),
        'Recall (macro)': round(report['macro avg']['recall'], 4),
        'F1-Score (macro)': round(report['macro avg']['f1-score'], 4),
        'Train Time (s)': round(res['time'], 2)
    })

summary_df = pd.DataFrame(summary_rows).set_index('Model')
summary_df = summary_df.sort_values('Accuracy', ascending=False)
print('Model Performance Summary:')
summary_df

In [ ]:
# Bar chart comparison
metrics = ['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1-Score (macro)']
x = np.arange(len(summary_df))
width = 0.2
palette = ['#1976d2', '#43a047', '#f57c00', '#ab47bc']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (metric, color) in enumerate(zip(metrics, palette)):
    ax.bar(x + i * width, summary_df[metric], width, label=metric, color=color, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(summary_df.index, rotation=15, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Classifier Performance Comparison (TF-IDF Features)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
plt.tight_layout()
plt.show()

## 9. Best Model — Detailed Analysis

We select the best model by **macro F1-score** (which is more informative than accuracy for imbalanced classes) and examine its confusion matrix and per-class metrics.

In [ ]:
best_name = summary_df['F1-Score (macro)'].idxmax()
best_res  = results[best_name]
y_pred_best = best_res['y_pred']

print(f'Best model: {best_name}')
print(f'Accuracy  : {best_res["accuracy"]:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_best, target_names=['negative', 'neutral', 'positive']))

In [ ]:
# Confusion matrix
labels = ['negative', 'neutral', 'positive']
cm = confusion_matrix(y_test, y_pred_best, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'{best_name}\nConfusion Matrix (Counts)', fontweight='bold')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm.round(3), display_labels=labels)
disp_norm.plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title(f'{best_name}\nConfusion Matrix (Normalized)', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Per-class F1 heatmap across all models
classes = ['negative', 'neutral', 'positive']
f1_matrix = []

for name in summary_df.index:
    row = []
    for cls in classes:
        row.append(round(results[name]['report'][cls]['f1-score'], 4))
    f1_matrix.append(row)

f1_df = pd.DataFrame(f1_matrix, index=summary_df.index, columns=classes)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(f1_df, annot=True, fmt='.3f', cmap='YlGn', ax=ax,
            linewidths=0.5, vmin=0.5, vmax=1.0)
ax.set_title('Per-Class F1-Score by Classifier', fontsize=13, fontweight='bold')
ax.set_xlabel('Sentiment Class')
ax.set_ylabel('Classifier')
plt.tight_layout()
plt.show()

## 10. Sentiment Analysis on Original Reviews

In [ ]:
# Apply the best model to the full dataset to get predicted sentiment
best_model = best_res['model']

X_full = tfidf.transform(df['final_text'])

if best_name == 'XGBoost':
    y_full_enc = best_model.predict(X_full)
    df['predicted_sentiment'] = le.inverse_transform(y_full_enc)
else:
    df['predicted_sentiment'] = best_model.predict(X_full)

# Agreement rate between label and prediction
agreement = (df['sentiment'] == df['predicted_sentiment']).mean()
print(f'Label vs Prediction agreement rate: {agreement:.2%}')
print()
print('Predicted sentiment distribution:')
print(df['predicted_sentiment'].value_counts())

In [ ]:
# Sentiment trend over time
df['at'] = pd.to_datetime(df['at'], errors='coerce')
df['year_month'] = df['at'].dt.to_period('M')

trend = df.groupby(['year_month', 'sentiment']).size().unstack(fill_value=0)
trend.index = trend.index.astype(str)

fig, ax = plt.subplots(figsize=(16, 6))
for sentiment, color in colors_sent.items():
    if sentiment in trend.columns:
        ax.plot(trend.index, trend[sentiment], label=sentiment,
                color=color, linewidth=2, marker='o', markersize=4)

ax.set_xlabel('Month')
ax.set_ylabel('Number of Reviews')
ax.set_title('Sentiment Trend Over Time — GoPay Reviews', fontsize=13, fontweight='bold')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Sample predictions — correct and incorrect
correct   = df[df['sentiment'] == df['predicted_sentiment']].sample(5, random_state=42)
incorrect = df[df['sentiment'] != df['predicted_sentiment']].sample(5, random_state=42)

print('=== Correctly Classified Samples ===')
display(correct[['content', 'sentiment', 'predicted_sentiment']].reset_index(drop=True))

print('\n=== Misclassified Samples ===')
display(incorrect[['content', 'sentiment', 'predicted_sentiment']].reset_index(drop=True))

## 11. Save Results

In [ ]:
# Save final dataframe with predictions
cols_to_save = ['content', 'score', 'at', 'sentiment', 'predicted_sentiment', 'final_text']
df_sentiment = df[cols_to_save].copy()

output_path = '/content/drive/MyDrive/Tugas 1/Dataset/gopay_reviews_sentiment.csv'
df_sentiment.to_csv(output_path, index=False)

print(f'Saved: {output_path}')
print(f'Shape: {df_sentiment.shape}')
df_sentiment.head(5)

## 12. Summary

| Metric | Value |
|--------|-------|
| Total reviews analysed | (fill in) |
| Feature extraction | TF-IDF (max 10K features, unigrams+bigrams) |
| Train / Test split | 80% / 20% (stratified) |
| Classifiers trained | 5 (Linear SVM, LR, Naive Bayes, XGBoost, RF) |
| Best model | (fill in after running) |
| Best Accuracy | (fill in) |
| Best Macro F1 | (fill in) |